# DURCL for RL-Based LLM Post-Training

This notebook accompanies the DSAA3053 course project report. It reproduces the checked-in numerical summaries and visualizations used in the paper. It does not rerun expensive RL training; the training outputs are taken from the repository CSV/JSON summaries.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
print(ROOT)

## Method note: from DUMP to DURCL

Original DUMP tracks a distribution-level sliding window of recent absolute advantages. DURCL changes this in two ways: it uses an exponential-decay advantage state to reduce retained history and smooth updates, and it adds micro-bucket rebucketing so the cluster structure can adapt when Math and Zebra drift at different learning rates. The mixed Math+Zebra setting is treated as a non-stationary bandit because useful clusters change as GRPO group-relative advantages drift during training.

## Step-200 Math+Zebra comparison

The clearest positive result in the current repository is the step-200 comparison between random sampling and DURCL sampling without rebucketing.

In [ ]:
mz = pd.read_csv(ROOT / 'experiments/results/math_zebra_2data/baseline_vs_norebucket_metrics.csv')
display(mz)

step200 = mz[mz['step'] == 200].set_index('run')
delta = step200.loc['scheduler_no_rebucket', ['math_train', 'zebra_train']] - step200.loc['baseline_random', ['math_train', 'zebra_train']]
display(delta.to_frame('scheduler_minus_random'))

## UCB score drift

The UCB trace shows that the sampler is active rather than fixed. The cluster ranking shifts from C0/C1 early in training to C3/C4 by step 200.

In [ ]:
ucb = pd.DataFrame({
    'step': [60, 80, 100, 120, 140, 160, 180, 200],
    'C0': [0.329, 0.250, 0.190, 0.135, 0.072, 0.101, 0.034, 0.081],
    'C1': [0.301, 0.264, 0.210, 0.189, 0.148, 0.141, 0.106, 0.101],
    'C2': [0.259, 0.243, 0.250, 0.204, 0.176, 0.184, 0.197, 0.182],
    'C3': [0.238, 0.252, 0.256, 0.275, 0.275, 0.233, 0.225, 0.295],
    'C4': [0.230, 0.235, 0.220, 0.216, 0.257, 0.255, 0.264, 0.253],
})
display(ucb)
display(pd.DataFrame({
    'step_60_rank': ucb.iloc[0, 1:].sort_values(ascending=False).index.tolist(),
    'step_200_rank': ucb.iloc[-1, 1:].sort_values(ascending=False).index.tolist(),
}))

## Initial accuracy test

These values diagnose why pure difficulty-label initialization is coarse. Zebra is not strictly monotone by nominal difficulty, while Math and Zebra have different accuracy scales.

In [ ]:
init_acc = pd.DataFrame({
    'difficulty': ['d1', 'd2', 'd3', 'd4', 'd5'],
    'zebra': [0.30, 0.20, 0.25, 0.15, None],
    'math': [0.70, 0.60, 0.35, 0.35, 0.10],
})
display(init_acc)

## Generate and display paper figures

The script uses `reportlab` to create vector PDFs and, when ImageMagick is available, PNG copies for notebook display.

In [ ]:
subprocess.run([sys.executable, 'paper/scripts/make_figures.py'], check=True)

for name in [
    'fig_method_pipeline',
    'fig_step200_math_zebra',
    'fig_ucb_score_drift',
    'fig_initial_accuracy_profile',
    'fig_rebucket_composition',
    'fig_inferred_transition_matrix',
    'fig_toy_rebucket_guardrails',
]:
    png = ROOT / 'paper/figures' / f'{name}.png'
    if png.exists():
        print(name)
        display(Image(filename=str(png)))
    else:
        print(f'PNG missing for {name}; PDF was still generated.')

## Experiment 2 accuracy-initialized result

The second experiment compares no-rebucket vs rebucket under accuracy-based initialization. Both validation results are now available: rebucketing improves Math over the no-rebucket variant and matches Zebra at step 200.

In [ ]:
exp2_acc_init = pd.DataFrame([
    {'run': 'acc_init_no_rebucket', 'step': 100, 'math': 0.480, 'zebra': 0.263},
    {'run': 'acc_init_no_rebucket', 'step': 200, 'math': 0.500, 'zebra': 0.300},
    {'run': 'acc_init_rebucket', 'step': 200, 'math': 0.540, 'zebra': 0.300},
])
display(exp2_acc_init)

## Re-bucketing interpretation

The current re-bucketing evidence supports a structural-correction claim more strongly than an overall final-accuracy claim. The composition drift figure shows that re-bucketing changes task mixture across clusters, which is both useful diagnostic evidence and a confound for causal interpretation.